# 02 — Indexar Chunks en Qdrant con Ollama

Genera embeddings con un modelo configurable de Ollama e indexa los chunks en Qdrant.
Cada combinación de estrategia de chunking + modelo de embedding crea su propia colección.

**Prerequisito:** ejecutar `01_prepare_chunks_from_jsonl_corpus.ipynb` con la misma estrategia para generar `chunks_<strategy>.jsonl`.

**Credenciales:** la API key de Qdrant se lee desde la variable de entorno `QDRANT_API_KEY`:
```bash
export QDRANT_API_KEY=tu_clave
```

| Parámetro | Valor |
|-----------|-------|
| `CHUNK_STRATEGY` | `"fixed"` \| `"recursive"` \| `"markdown"` \| `"section"` \| `"semantic"` |
| `EMBED_MODEL` | nombre del modelo en Ollama (ej. `"nomic-embed-text"`, `"qwen3-embedding"`, `"avr/sfr-embedding-mistral"`...) |
| Colección generada | `altia_rag_{slug(EMBED_MODEL)}_{CHUNK_STRATEGY}` |
| Dimensión | auto-detectada a partir del primer embedding de prueba |
| Distancia | Cosine |
| Batch size | 16 |

In [15]:
%pip install qdrant-client tqdm requests --quiet

Note: you may need to restart the kernel to use updated packages.


In [16]:
import getpass
import json
import os
import re
import time
import uuid
from pathlib import Path

import requests
from tqdm.auto import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


In [ ]:
# ── Configuración ──────────────────────────────────────────────────────────────
# Tier 1: "fixed_256" | "fixed_512" | "fixed_1024" | "recursive_200" | "recursive_400"
# Tier 2: "markdown" | "section"
# Tier 3: "semantic"
CHUNK_STRATEGY = "semantic"

CHUNKS_DIR   = "/home/coder/ia-testing/rageval/data/02_intermediate"
OLLAMA_URL   = "http://localhost:11434"
EMBED_MODEL  = "nomic-embed-text"
QDRANT_URL   = "https://endorqdrant.altia.es:443"
BATCH_SIZE   = 16
RELOAD_EVERY = 100

# parent_text puede ser enorme en secciones largas (IT_04_06, etc.).
# Qdrant tiene límite de 32MB por request; truncamos a 16000 chars,
# proporcional al num_ctx=16384 del LLM generador en nb03.
PARENT_TEXT_MAX_CHARS = 16000

In [ ]:
# Derivación — corre DESPUÉS del override de papermill
import re as _re

def _model_slug(model: str) -> str:
    """avr/sfr-embedding-mistral -> sfrembeddingmistral"""
    name = model.split("/")[-1]                    # quita prefijo usuario/org
    name = _re.sub(r"[^a-zA-Z0-9]+", "", name)      # quita guiones, puntos...
    return name.lower()

CHUNKS_PATH     = f"{CHUNKS_DIR}/chunks_{CHUNK_STRATEGY}.jsonl"
COLLECTION_NAME = f"altia_rag_{_model_slug(EMBED_MODEL)}_{CHUNK_STRATEGY}"

print(f"Estrategia  : {CHUNK_STRATEGY}")
print(f"Embedding   : {EMBED_MODEL}")
print(f"Chunks path : {CHUNKS_PATH}")
print(f"Colección   : {COLLECTION_NAME}")

## 1. Leer API key desde variable de entorno

In [19]:
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY") or getpass.getpass("QDRANT_API_KEY: ")
print("QDRANT_API_KEY cargada correctamente.")

QDRANT_API_KEY cargada correctamente.


## 2. Verificar Ollama y modelo de embedding

In [ ]:
resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10)
resp.raise_for_status()
models = [m["name"] for m in resp.json().get("models", [])]
print(f"Modelos disponibles en Ollama: {models}")

if not any(EMBED_MODEL in m for m in models):
    print(f"\n⚠️  Modelo '{EMBED_MODEL}' no encontrado.")
    print(f"   Descárgalo con: ollama pull {EMBED_MODEL}")
else:
    print(f"\n✓  Modelo '{EMBED_MODEL}' disponible.")

## 3. Función de embedding y test de dimensión

In [ ]:
import re as _re

# Límite de caracteres por texto a embeber. nomic-embed-text usa 2048 tokens
# de contexto (~8000 chars en español); el resto de modelos del barrido
# (qwen3-embedding, gte-qwen2, Octen, sfr-embedding-mistral) tienen contextos
# de 32K-125K tokens. 6000 chars cubre con margen a los 5 sin lógica condicional
# por modelo (los chunks indexados miden ≤450 palabras ≈ 3000 chars).
_EMBED_MAX_CHARS = 6000

def _sanitize(text: str) -> str:
    return _re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

def embed_text(text: str, max_retries: int = 3) -> list[float]:
    clean = _sanitize(text).strip()[:_EMBED_MAX_CHARS]
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                f"{OLLAMA_URL}/api/embed",
                json={"model": EMBED_MODEL, "input": clean},
                timeout=120,
            )
            resp.raise_for_status()
            return resp.json()["embeddings"][0]
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5)
            else:
                raise

In [ ]:
test_vec = embed_text("proceso de calidad")
EMBEDDING_DIM = len(test_vec)
print(f"Dimensión del embedding: {EMBEDDING_DIM}")

assert EMBEDDING_DIM > 0, "Embedding vacío — revisa que el modelo esté cargado en Ollama"
print("✓  Embedding OK.")
time.sleep(3)


## 4. Conectar a Qdrant y crear colección

In [23]:
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, check_compatibility=False)

existing = [c.name for c in client.get_collections().collections]
print(f"Colecciones existentes: {existing}")

if COLLECTION_NAME in existing:
    info = client.get_collection(COLLECTION_NAME)
    print(f"\nColección '{COLLECTION_NAME}' ya existe → {info.points_count} puntos indexados.")
    print("Si quieres reindexar desde cero, bórrala con: client.delete_collection(COLLECTION_NAME)")
else:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    print(f"\nColección '{COLLECTION_NAME}' creada (dim={EMBEDDING_DIM}, cosine).")

Colecciones existentes: ['altia_rag_fixed_1024', 'altia_rag_fixed_256', 'altia_rag_fixed_512', 'altia_rag_markdown', 'altia_rag_recursive_200', 'altia_rag_recursive_400', 'altia_rag_section', 'test1']

Colección 'altia_rag_fixed_512' ya existe → 4938 puntos indexados.
Si quieres reindexar desde cero, bórrala con: client.delete_collection(COLLECTION_NAME)


## 5. Cargar chunks

In [24]:
chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print(f"Chunks cargados: {len(chunks)}")
print(f"Ejemplo de metadata: {list(chunks[0]['metadata'].keys())}")

Chunks cargados: 4938
Ejemplo de metadata: ['chunk_id', 'doc_id', 'doc_code', 'file_name', 'chunk_strategy', 'chunk_size', 'chunk_overlap', 'doc_type', 'doc_title', 'doc_version', 'doc_date', 'process_area', 'doc_process_family', 'doc_total_words', 'doc_in_degree', 'doc_out_degree', 'parent_process_file', 'parent_process_title', 'sibling_its', 'chunk_index', 'chunk_total', 'word_count', 'char_count', 'section_title', 'breadcrumb', 'section_level', 'has_table', 'has_steps', 'has_roles', 'has_definitions', 'has_requirements', 'has_exceptions']


## 6. Indexar en Qdrant

In [ ]:
def maybe_reload_model(batch_num: int) -> None:
    """Descarga el modelo de Ollama cada RELOAD_EVERY batches para liberar VRAM."""
    if batch_num > 0 and batch_num % RELOAD_EVERY == 0:
        requests.post(
            f"{OLLAMA_URL}/api/embed",
            json={"model": EMBED_MODEL, "input": "ping", "keep_alive": 0},
            timeout=30,
        )
        time.sleep(3)

In [ ]:
points_uploaded = 0
errors = []

for batch_num, i in enumerate(tqdm(range(0, len(chunks), BATCH_SIZE), desc="Indexando batches")):
    maybe_reload_model(batch_num)
    batch = chunks[i : i + BATCH_SIZE]
    points = []

    for chunk in batch:
        try:
            vector = embed_text(chunk["text"])
            meta = dict(chunk["metadata"])
            if meta.get("parent_text"):
                meta["parent_text"] = meta["parent_text"][:PARENT_TEXT_MAX_CHARS]
            points.append(
                PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vector,
                    payload={"page_content": chunk["text"], **meta},
                )
            )
        except Exception as e:
            errors.append({"chunk_id": chunk["metadata"].get("chunk_id"), "error": str(e)})

    if points:
        client.upsert(collection_name=COLLECTION_NAME, points=points)
        points_uploaded += len(points)

print(f"Puntos subidos: {points_uploaded}")
if errors:
    print(f"Errores: {len(errors)}")
    for e in errors[:5]:
        print(f"  {e}")

## 7. Verificación final

In [27]:
info = client.get_collection(COLLECTION_NAME)
print(f"Colección    : {COLLECTION_NAME}")
print(f"Vectores     : {info.points_count}")
print(f"Dimensión    : {info.config.params.vectors.size}")
print(f"Distancia    : {info.config.params.vectors.distance}")

Colección    : altia_rag_fixed_512
Vectores     : 9876
Dimensión    : 768
Distancia    : Cosine


In [28]:
query = "¿Cuál es el coste de la aplicación DAVdroid?"
query_vec = embed_text(query)

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=5,
).points

W = 62
FIELD_ORDER = [
    "chunk_id", "doc_id", "doc_code", "file_name",
    "chunk_strategy", "chunk_size", "chunk_overlap",
    "doc_type", "doc_title", "doc_version", "doc_date", "process_area",
    "doc_process_family", "doc_total_words", "doc_in_degree", "doc_out_degree",
    "parent_process_file", "parent_process_title", "sibling_its",
    "chunk_index", "chunk_total", "word_count", "char_count",
    "section_title", "section_level", "breadcrumb",
]

print(f"Top 5 resultados para: '{query}'\n")
for r in results:
    p = r.payload
    print("═" * W)
    print(f"  qdrant_id    : {r.id}")
    print(f"  score        : {r.score:.4f}")
    print()
    for key in FIELD_ORDER:
        if key in p:
            print(f"  {key:<22}: {p[key]}")
    flags = {k: v for k, v in p.items() if k.startswith("has_")}
    if flags:
        print(f"\n  flags:")
        items = list(flags.items())
        for i in range(0, len(items), 2):
            left  = f"{items[i][0]:<20}: {str(items[i][1]):<6}"
            right = f"{items[i+1][0]:<20}: {items[i+1][1]}" if i + 1 < len(items) else ""
            print(f"    {left}  {right}")
    extra = {k: v for k, v in p.items() if k not in FIELD_ORDER and not k.startswith("has_") and k != "page_content"}
    for k, v in extra.items():
        print(f"  {k:<22}: {v}")
    print(f"\n  page_content:\n{p.get('page_content', '')}")
    print()
print("═" * W)


Top 5 resultados para: '¿Cuál es el coste de la aplicación DAVdroid?'

══════════════════════════════════════════════════════════════
  qdrant_id    : 1ed30b6c-1535-4d69-9052-62cdaa227d77
  score        : 0.7271

  chunk_id              : IT_04_03_chunk_023
  doc_id                : IT_04_03
  doc_code              : IT_04_03
  file_name             : IT_04_03_Planes_de_formacion_y_formacion_en_el_marco_de_nuestros_proyectos.odt.md
  chunk_strategy        : fixed_512
  chunk_size            : 512
  chunk_overlap         : 64
  doc_type              : instruccion_tecnica
  doc_title             : Roles y Responsabilidades
  doc_version           : 30.0
  doc_date              : 05/10/2018
  process_area          : soporte_admin
  doc_process_family    : procesos_soporte_administrativo
  doc_total_words       : 11619
  doc_in_degree         : 8
  doc_out_degree        : 24
  parent_process_file   : P_04_Gestion_RRHH.odt.md
  parent_process_title  : Procedimientos relacionados
  sibling_i